# 注意力机制 (Attention Mechanisms)

> 参考：[动手学深度学习 v2 第10章](https://zh-v2.d2l.ai/chapter_attention-mechanisms/index.html)

**核心思想**：区分*自主性提示*（查询 Query）与*非自主性提示*（键 Key），通过注意力汇聚对值（Value）进行加权聚合。

## 目录
1. [注意力机制概述](#1-注意力机制概述)
2. [注意力汇聚：Nadaraya-Watson 核回归](#2-注意力汇聚nadaraya-watson-核回归)
3. [注意力评分函数](#3-注意力评分函数)
4. [Bahdanau 注意力（seq2seq+注意力）](#4-bahdanau-注意力)
5. [多头注意力](#5-多头注意力)
6. [自注意力与位置编码](#6-自注意力与位置编码)
7. [Transformer](#7-transformer)


## 0. 环境配置

In [ ]:
import torch
import torch.nn as nn
import math
import numpy as np
import matplotlib.pyplot as plt

# 安装 d2l（如未安装）
# !pip install d2l

import d2l.torch as d2l

device = d2l.try_gpu()
print(f"使用设备：{device}")
print(f"PyTorch 版本：{torch.__version__}")


## 1. 注意力机制概述

### 1.1 生物学启发

人类的视觉注意力是**有限的**稀缺资源：
- **非自主性提示**（Non-volitional cue）：基于突显性（显著性）自动吸引注意，如一张红色图片中的红色物体。
- **自主性提示**（Volitional cue）：基于任务主动搜索，如寻找一本书。

### 1.2 查询、键、值框架

$$	ext{Attention}(\mathbf{q}, \{(\mathbf{k}_1,\mathbf{v}_1), \ldots, (\mathbf{k}_m,\mathbf{v}_m)\}) = \sum_{i=1}^{m} lpha(\mathbf{q}, \mathbf{k}_i)\, \mathbf{v}_i$$

| 组件 | 类比 | 说明 |
|---|---|---|
| Query $\mathbf{q}$ | 自主性提示 | 当前任务/解码器状态 |
| Key $\mathbf{k}_i$ | 非自主性提示 | 每个位置的表示 |
| Value $\mathbf{v}_i$ | 感官输入 | 每个位置的实际内容 |
| $\alpha(\mathbf{q},\mathbf{k}_i)$ | 注意力权重 | 非负，和为1（softmax后） |

注意力权重 $\alpha$ 通过**评分函数** $a(\mathbf{q}, \mathbf{k}_i)$ 经 softmax 归一化得到。


## 2. 注意力汇聚：Nadaraya-Watson 核回归

### 2.1 三种回归方式

**平均汇聚（Average Pooling）**——最简单的基线：
$$f(x) = \frac{1}{n}\sum_{i=1}^{n} y_i$$

**Nadaraya-Watson 核回归（1964）**——非参数注意力汇聚：
$$f(x) = \sum_{i=1}^{n} \frac{K(x - x_i)}{\sum_{j=1}^{n} K(x - x_j)} y_i = \sum_{i=1}^{n} \alpha(x, x_i)\, y_i$$

使用**高斯核** $K(u) = \frac{1}{\sqrt{2\pi}} \exp\!\left(-\frac{u^2}{2}\right)$，得到：
$$f(x) = \sum_{i=1}^{n} \mathrm{softmax}\!\left(-\frac{1}{2}(x - x_i)^2\right) y_i$$

**参数化版本**——可学习的带宽参数 $w$：
$$f(x) = \sum_{i=1}^{n} \mathrm{softmax}\!\left(-\frac{1}{2}\bigl((x-x_i)w\bigr)^2\right) y_i$$

### 2.2 非参数 NW 回归

查询距离越近的键（$x_i$）获得更大的注意力权重。


In [ ]:
# 生成训练数据
n_train = 50
x_train, _ = torch.sort(torch.rand(n_train) * 5)
y_train = torch.sin(x_train) + torch.normal(0, 0.5, (n_train,))
x_test = torch.arange(0, 5, 0.1)
y_truth = torch.sin(x_test)

# ── 非参数 Nadaraya-Watson 核回归 ──
X_repeat = x_test.repeat_interleave(n_train).reshape((-1, n_train))
attention_weights = nn.functional.softmax(-(X_repeat - x_train) ** 2 / 2, dim=1)
y_hat_nw = torch.matmul(attention_weights, y_train)

plt.figure(figsize=(8, 3))
plt.plot(x_test, y_truth, label='真实函数', linewidth=2)
plt.plot(x_test, y_hat_nw, '--', label='NW 核回归')
plt.scatter(x_train, y_train, s=15, alpha=0.5, label='训练数据')
plt.legend(); plt.title('非参数 Nadaraya-Watson 核回归'); plt.show()


### 2.3 参数化 NW 核回归（可学习权重）

In [ ]:
class NWKernelRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.rand(1, requires_grad=True))

    def forward(self, queries, keys, values):
        # queries: (nq,), keys: (nq, n_train), values: (nq, n_train)
        queries = queries.repeat_interleave(keys.shape[1]).reshape(-1, keys.shape[1])
        self.attention_weights = nn.functional.softmax(
            -((queries - keys) * self.w) ** 2 / 2, dim=1)
        return torch.bmm(self.attention_weights.unsqueeze(1),
                         values.unsqueeze(-1)).reshape(-1)

# 准备数据
keys   = x_train.repeat(len(x_test), 1)   # (len(x_test), n_train)
values = y_train.repeat(len(x_test), 1)

net = NWKernelRegression()
loss_fn = nn.MSELoss(reduction='none')
trainer = torch.optim.SGD(net.parameters(), lr=0.5)

for epoch in range(5):
    trainer.zero_grad()
    l = loss_fn(net(x_test, keys, values), y_truth)
    l.sum().backward()
    trainer.step()
    print(f'epoch {epoch+1}, loss {l.sum():.4f}, w={net.w.item():.4f}')

# 可视化注意力权重
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(net.attention_weights.detach().numpy(), aspect='auto')
axes[0].set_title('注意力权重热图'); axes[0].set_xlabel('训练样本'); axes[0].set_ylabel('查询')
axes[1].plot(x_test, y_truth, label='真实函数')
axes[1].plot(x_test, net(x_test, keys, values).detach(), '--', label='参数化NW')
axes[1].scatter(x_train, y_train, s=15, alpha=0.5)
axes[1].legend(); axes[1].set_title('参数化 NW 核回归拟合')
plt.tight_layout(); plt.show()


## 3. 注意力评分函数

注意力权重通过**评分函数** $a(\mathbf{q}, \mathbf{k})$ 得到：

$$\alpha(\mathbf{q}, \mathbf{k}_i) = \frac{\exp(a(\mathbf{q}, \mathbf{k}_i))}{\sum_j \exp(a(\mathbf{q}, \mathbf{k}_j))}$$

### 3.1 加性注意力（Additive Attention）

适用于**查询和键维度不同**的情形：

$$a(\mathbf{q}, \mathbf{k}) = \mathbf{w}_v^\top \tanh(\mathbf{W}_q \mathbf{q} + \mathbf{W}_k \mathbf{k}) \in \mathbb{R}$$

其中 $\mathbf{W}_q \in \mathbb{R}^{h \times q}$，$\mathbf{W}_k \in \mathbb{R}^{h \times k}$，$\mathbf{w}_v \in \mathbb{R}^h$。

### 3.2 缩放点积注意力（Scaled Dot-Product Attention）

适用于**查询和键维度相同**（均为 $d$）的情形，计算效率更高：

$$a(\mathbf{q}, \mathbf{k}) = \frac{\mathbf{q}^\top \mathbf{k}}{\sqrt{d}}$$

批量矩阵形式（$n$ 个查询，$m$ 个键值对）：

$$\mathrm{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right)\mathbf{V} \in \mathbb{R}^{n \times v}$$

> **缩放的意义**：防止点积随 $d$ 增大而过大，导致 softmax 梯度消失。

### 3.3 遮蔽 softmax（Masked Softmax）

对填充（padding）位置赋予极小值（$-10^6$），使其注意力权重趋近于零。


In [ ]:
# ── 遮蔽 Softmax ──
def masked_softmax(X, valid_lens):
    '''对 X 做 softmax，但遮蔽超出 valid_lens 的位置'''
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    shape = X.shape
    if valid_lens.dim() == 1:
        valid_lens = torch.repeat_interleave(valid_lens, shape[1])
    else:
        valid_lens = valid_lens.reshape(-1)
    # 超出有效长度的位置填充为极小值
    X = d2l.sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
    return nn.functional.softmax(X.reshape(shape), dim=-1)

# 示例：batch=2, 2个查询, 4个键
X = torch.rand(2, 2, 4)
valid_lens = torch.tensor([2, 3])
weights = masked_softmax(X, valid_lens)
print("遮蔽 softmax 权重（每行和为1，超出部分趋近0）:")
print(weights)


### 3.4 加性注意力实现

In [ ]:
class AdditiveAttention(nn.Module):
    '''加性注意力：适用于 q_dim != k_dim 的情形'''
    def __init__(self, key_size, query_size, num_hiddens, dropout):
        super().__init__()
        self.W_k = nn.Linear(key_size, num_hiddens, bias=False)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=False)
        self.w_v = nn.Linear(num_hiddens, 1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens):
        # queries: (B, nq, q_dim), keys: (B, nk, k_dim)
        queries = self.W_q(queries)  # (B, nq, h)
        keys    = self.W_k(keys)     # (B, nk, h)
        # 广播相加: (B, nq, 1, h) + (B, 1, nk, h) -> (B, nq, nk, h)
        features = queries.unsqueeze(2) + keys.unsqueeze(1)
        features = torch.tanh(features)
        scores = self.w_v(features).squeeze(-1)  # (B, nq, nk)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)

# 示例
queries = torch.normal(0, 1, (2, 1, 20))   # batch=2, 1 query, dim=20
keys    = torch.ones((2, 10, 2))           # 10 keys, dim=2
values  = torch.arange(40, dtype=torch.float32).reshape(1, 10, 4).repeat(2, 1, 1)
valid_lens = torch.tensor([2, 6])

attention = AdditiveAttention(key_size=2, query_size=20, num_hiddens=8, dropout=0.1)
attention.eval()
output = attention(queries, keys, values, valid_lens)
print(f"加性注意力输出形状: {output.shape}")  # (2, 1, 4)
print(f"注意力权重: {attention.attention_weights}")


### 3.5 缩放点积注意力实现

In [ ]:
class DotProductAttention(nn.Module):
    '''缩放点积注意力：适用于 q_dim == k_dim 的情形'''
    def __init__(self, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens=None):
        # queries/keys/values: (B, n, d), (B, m, d), (B, m, v)
        d = queries.shape[-1]
        # scores: (B, n, m)
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)

# 示例
queries = torch.normal(0, 1, (2, 1, 2))
dot_attn = DotProductAttention(dropout=0.5)
dot_attn.eval()
output = dot_attn(queries, keys, values, valid_lens)
print(f"缩放点积注意力输出形状: {output.shape}")


## 4. Bahdanau 注意力

### 4.1 动机：突破信息瓶颈

经典 seq2seq 的缺陷：编码器将**整个输入序列压缩成固定长度**的上下文向量 $\mathbf{c}$，解码时所有步骤共享同一个 $\mathbf{c}$，对长序列表达能力有限。

**Bahdanau（2015）的解决方案**：为每个解码步骤动态计算不同的上下文向量。

### 4.2 数学原理

解码步骤 $t'$ 的上下文向量：

$$\mathbf{c}_{t'} = \sum_{t=1}^{T} \alpha(\mathbf{s}_{t'-1}, \mathbf{h}_t)\, \mathbf{h}_t$$

- $\mathbf{s}_{t'-1}$：解码器上一步隐藏状态（Query）
- $\mathbf{h}_t$：编码器第 $t$ 步隐藏状态（Key 和 Value）
- $\alpha$：加性注意力权重

与标准 seq2seq 对比：

| | 标准 seq2seq | Bahdanau |
|---|---|---|
| 上下文向量 | 固定，编码器最后隐状态 | 动态，每步重新计算 |
| 对齐 | 隐式 | 显式学习 |
| 可解释性 | 弱 | 可视化注意力权重 |


In [ ]:
# Bahdanau 注意力解码器（使用 d2l 框架）
class Seq2SeqAttentionDecoder(d2l.AttentionDecoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super().__init__()
        self.attention = d2l.AdditiveAttention(
            num_hiddens, num_hiddens, num_hiddens, dropout)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(
            embed_size + num_hiddens, num_hiddens,
            num_layers, dropout=dropout)
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, enc_valid_lens, *args):
        # enc_outputs: (seq_len, batch, num_hiddens)
        # outputs -> (batch, seq_len, num_hiddens)
        outputs, hidden_state = enc_outputs
        return (outputs.permute(1, 0, 2), hidden_state, enc_valid_lens)

    def forward(self, X, state):
        # X: (batch, num_steps) -> (num_steps, batch, embed_size)
        enc_outputs, hidden_state, enc_valid_lens = state
        X = self.embedding(X).permute(1, 0, 2)
        outputs, self._attention_weights = [], []
        for x in X:
            # query: (batch, 1, num_hiddens) <- 上一步 hidden state
            query = torch.unsqueeze(hidden_state[-1], dim=1)
            # context: (batch, 1, num_hiddens)
            context = self.attention(query, enc_outputs, enc_outputs, enc_valid_lens)
            # 拼接 embedding 和 context
            x = torch.cat([context, torch.unsqueeze(x, dim=1)], dim=-1)
            out, hidden_state = self.rnn(x.permute(1, 0, 2), hidden_state)
            outputs.append(out)
            self._attention_weights.append(self.attention.attention_weights)
        outputs = self.dense(torch.cat(outputs, dim=0))
        return outputs.permute(1, 0, 2), [enc_outputs, hidden_state, enc_valid_lens]

    @property
    def attention_weights(self):
        return self._attention_weights

# 训练 Bahdanau 注意力 seq2seq（英译法）
embed_size, num_hiddens, num_layers, dropout = 32, 32, 2, 0.1
batch_size, num_steps = 64, 10
lr, num_epochs = 0.005, 250

train_iter, src_vocab, tgt_vocab = d2l.load_data_nmt(batch_size, num_steps)
encoder = d2l.Seq2SeqEncoder(len(src_vocab), embed_size, num_hiddens, num_layers, dropout)
decoder = Seq2SeqAttentionDecoder(len(tgt_vocab), embed_size, num_hiddens, num_layers, dropout)
net = d2l.EncoderDecoder(encoder, decoder)
d2l.train_seq2seq(net, train_iter, lr, num_epochs, tgt_vocab, device)


In [ ]:
# 预测并可视化注意力权重
engs = ['go .', 'i lost .', 'he's calm .', 'i'm home .']
fras = ['va !',  'j'ai perdu .', 'il est calme .', 'je suis chez moi .']

for eng, fra in zip(engs, fras):
    translation, dec_attention_weight_seq = d2l.predict_seq2seq(
        net, eng, src_vocab, tgt_vocab, num_steps, device, save_attention_weights=True)
    print(f'{eng} => {translation}')

# 可视化最后一个句子的注意力
attention_weights = torch.cat(
    [step[0][0][0] for step in dec_attention_weight_seq], 0
).reshape((1, 1, -1, num_steps))

d2l.show_heatmaps(
    attention_weights[:, :, :, :len(engs[-1].split()) + 1].cpu(),
    xlabel='Key 位置', ylabel='Query 位置',
    titles=['注意力权重'], figsize=(7, 3.5))


## 5. 多头注意力

### 5.1 动机

单一注意力机制只能在一个表示子空间中捕获依赖关系。**多头注意力**让模型同时在多个子空间中学习不同的注意力模式。

### 5.2 数学原理

**第 $i$ 个注意力头**（独立的线性投影 + 注意力）：

$$\mathbf{h}_i = f\bigl(\mathbf{W}_i^{(q)}\mathbf{q},\; \mathbf{W}_i^{(k)}\mathbf{k},\; \mathbf{W}_i^{(v)}\mathbf{v}\bigr) \in \mathbb{R}^{p_v}$$

**拼接并线性变换**：

$$\mathbf{W}_o \begin{bmatrix}\mathbf{h}_1 \\ \vdots \\ \mathbf{h}_h\end{bmatrix} \in \mathbb{R}^{p_o}$$

其中 $\mathbf{W}_o \in \mathbb{R}^{p_o \times hp_v}$。

> **关键实现技巧**：通过 tensor reshape + permute 将所有头**并行计算**，避免 for 循环。

典型设置：$p_q = p_k = p_v = p_o / h$（每个头的维度 = 总维度 / 头数）。


In [ ]:
def transpose_qkv(X, num_heads):
    '''将 (batch, seq, num_hiddens) 变换为 (batch*num_heads, seq, num_hiddens/num_heads)'''
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    X = X.permute(0, 2, 1, 3)
    return X.reshape(-1, X.shape[2], X.shape[3])

def transpose_output(X, num_heads):
    '''transpose_qkv 的逆操作'''
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)


class MultiHeadAttention(nn.Module):
    def __init__(self, key_size, query_size, value_size,
                 num_hiddens, num_heads, dropout, bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.attention = DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size,   num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        # 投影后并行展开所有头
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys    = transpose_qkv(self.W_k(keys),    self.num_heads)
        values  = transpose_qkv(self.W_v(values),  self.num_heads)

        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens, self.num_heads, dim=0)

        output = self.attention(queries, keys, values, valid_lens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)

# 验证
num_hiddens, num_heads = 100, 5
attention = MultiHeadAttention(num_hiddens, num_hiddens, num_hiddens,
                               num_hiddens, num_heads, dropout=0.5)
attention.eval()
batch_size, num_queries, num_kvpairs = 2, 4, 6
valid_lens = torch.tensor([3, 2])
X = torch.ones((batch_size, num_queries,  num_hiddens))
Y = torch.ones((batch_size, num_kvpairs, num_hiddens))
out = attention(X, Y, Y, valid_lens)
print(f"多头注意力输出形状: {out.shape}")  # (2, 4, 100)


## 6. 自注意力与位置编码

### 6.1 自注意力

将同一组输入序列同时用作查询、键、值：

$$\text{self-attention}(\mathbf{X}) = \text{Attention}(\mathbf{X}, \mathbf{X}, \mathbf{X})$$

**复杂度对比**（序列长度 $n$，特征维度 $d$，卷积核大小 $k$）：

| 架构 | 计算复杂度 | 顺序操作数 | 最大路径长度 |
|---|---|---|---|
| 卷积（CNN） | $O(knd^2)$ | $O(1)$ | $O(n/k)$ |
| 循环（RNN） | $O(nd^2)$ | $O(n)$ | $O(n)$ |
| **自注意力** | $O(n^2d)$ | $O(1)$ | $O(1)$ |

自注意力的最大路径长度为 $O(1)$（任意两个位置直接相连），但对超长序列计算开销大（$O(n^2)$）。

### 6.2 位置编码

自注意力是**置换不变的**（position-agnostic），需要手动注入位置信息。

**正弦/余弦位置编码**：位置 $i$，维度 $2j$（偶数）和 $2j+1$（奇数）：

$$p_{i,2j} = \sin\!\left(\frac{i}{10000^{2j/d}}\right), \quad p_{i,2j+1} = \cos\!\left(\frac{i}{10000^{2j/d}}\right)$$

**相对位置性质**：位置 $i+\delta$ 可由位置 $i$ 通过仅依赖偏移量 $\delta$ 的线性（旋转）变换得到，使模型能学习相对位置关系。


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        # 预计算位置编码矩阵 P: (1, max_len, num_hiddens)
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1) / torch.pow(
            10000, torch.arange(0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)  # 偶数维度
        self.P[:, :, 1::2] = torch.cos(X)  # 奇数维度

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)

# 可视化位置编码
pe = PositionalEncoding(num_hiddens=6, dropout=0)
X = pe(torch.zeros(1, 100, 6))

plt.figure(figsize=(10, 4))
for i in range(6):
    plt.plot(X[0, :, i].detach(), label=f'dim {i}')
plt.title('正弦位置编码（前6个维度）')
plt.xlabel('序列位置'); plt.ylabel('编码值')
plt.legend(ncol=3); plt.show()


In [ ]:
# 可视化位置编码热图
encoding_dim, num_steps = 32, 60
pe = PositionalEncoding(encoding_dim, dropout=0)
X = pe(torch.zeros(1, num_steps, encoding_dim))
d2l.show_heatmaps(
    X[:, :, :].unsqueeze(0).permute(0, 3, 1, 2),
    xlabel='序列位置', ylabel='编码维度',
    figsize=(3.5, 4), cmap='Blues'
)


## 7. Transformer

### 7.1 架构概述

**Vaswani et al. (2017) "Attention Is All You Need"**

纯注意力架构，完全去除 RNN 和 CNN：

```
输入序列
  │
嵌入 × √d + 位置编码
  │
┌─────────────────────────────┐  ×N
│  EncoderBlock               │
│  ├─ 多头自注意力              │
│  └─ Add & Norm              │
│  ├─ Position-wise FFN       │
│  └─ Add & Norm              │
└─────────────────────────────┘
  │ enc_outputs
  ▼
┌─────────────────────────────┐  ×N
│  DecoderBlock               │
│  ├─ 掩蔽多头自注意力（causal） │
│  └─ Add & Norm              │
│  ├─ 多头交叉注意力            │
│  └─ Add & Norm              │
│  ├─ Position-wise FFN       │
│  └─ Add & Norm              │
└─────────────────────────────┘
  │
线性 + Softmax
  │
目标序列
```

### 7.2 核心子模块

**Add & Norm（残差 + 层归一化）**：
$$\text{output} = \text{LayerNorm}(\text{Dropout}(\text{sublayer}(X)) + X)$$

**LayerNorm vs BatchNorm**：LayerNorm 在特征维度上归一化（单样本），适合序列数据；BatchNorm 在批次维度上归一化。


In [ ]:
# ── 位置前馈网络（Position-wise FFN）──
class PositionWiseFFN(nn.Module):
    '''对序列每个位置独立应用相同的 2 层 MLP'''
    def __init__(self, ffn_num_input, ffn_num_hiddens, ffn_num_outputs):
        super().__init__()
        self.dense1 = nn.Linear(ffn_num_input, ffn_num_hiddens)
        self.relu   = nn.ReLU()
        self.dense2 = nn.Linear(ffn_num_hiddens, ffn_num_outputs)

    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))

# 验证：输出形状与输入相同（位置间独立）
ffn = PositionWiseFFN(4, 4, 8)
ffn.eval()
print(f"FFN 输出形状: {ffn(torch.ones(2, 3, 4)).shape}")  # (2, 3, 8)


In [ ]:
# ── Add & Norm ──
class AddNorm(nn.Module):
    def __init__(self, normalized_shape, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(normalized_shape)

    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)

# LayerNorm vs BatchNorm 直觉对比
ln = nn.LayerNorm(2)
bn = nn.BatchNorm1d(2)
X = torch.tensor([[1, 2], [2, 3]], dtype=torch.float32)
print(f"LayerNorm 输出（对每行特征归一化）:\n{ln(X)}")
print(f"BatchNorm 输出（对每列批次归一化）:\n{bn(X)}")


In [ ]:
# ── Encoder Block ──
class EncoderBlock(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout)
        self.addnorm1 = AddNorm(norm_shape, dropout)
        self.ffn      = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(norm_shape, dropout)

    def forward(self, X, valid_lens):
        # 多头自注意力 + 残差
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        # FFN + 残差
        return self.addnorm2(Y, self.ffn(Y))

# 验证
X = torch.ones((2, 100, 24))  # (batch, seq_len, num_hiddens)
valid_lens = torch.tensor([3, 2])
encoder_blk = EncoderBlock(24, 24, 24, 24, [24], 24, 48, 8, dropout=0.5)
encoder_blk.eval()
out = encoder_blk(X, valid_lens)
print(f"EncoderBlock 输出形状: {out.shape}")  # (2, 100, 24)


In [ ]:
# ── Transformer Encoder ──
class TransformerEncoder(d2l.Encoder):
    def __init__(self, vocab_size, key_size, query_size, value_size,
                 num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, dropout):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.embedding   = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential(*[
            EncoderBlock(key_size, query_size, value_size, num_hiddens,
                         norm_shape, ffn_num_input, ffn_num_hiddens, num_heads, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, X, valid_lens, *args):
        # 嵌入缩放 × √d_model，再加位置编码
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        self.attention_weights = [None] * len(self.blks)
        for i, blk in enumerate(self.blks):
            X = blk(X, valid_lens)
            self.attention_weights[i] = blk.attention.attention.attention_weights
        return X

# 验证
encoder = TransformerEncoder(200, 24, 24, 24, 24, [24], 24, 48, 8, num_layers=2, dropout=0.5)
encoder.eval()
X = torch.ones((2, 100), dtype=torch.long)
out = encoder(X, valid_lens)
print(f"TransformerEncoder 输出形状: {out.shape}")  # (2, 100, 24)


In [ ]:
# ── Decoder Block ──
class DecoderBlock(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads, dropout, i):
        super().__init__()
        self.i = i
        # 1. 掩蔽自注意力（causal，防止看到未来）
        self.attention1 = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout)
        self.addnorm1 = AddNorm(norm_shape, dropout)
        # 2. 编-解码器交叉注意力（encoder outputs 作为 K, V）
        self.attention2 = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout)
        self.addnorm2 = AddNorm(norm_shape, dropout)
        # 3. Position-wise FFN
        self.ffn      = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm3 = AddNorm(norm_shape, dropout)

    def forward(self, X, state):
        enc_outputs, enc_valid_lens = state[0], state[1]

        # 推理阶段逐步缓存 key-value
        if state[2][self.i] is None:
            key_values = X
        else:
            key_values = torch.cat((state[2][self.i], X), axis=1)
        state[2][self.i] = key_values

        # 训练时用掩码防止看到未来；推理时不需要（已逐步生成）
        if self.training:
            batch_size, num_steps, _ = X.shape
            dec_valid_lens = torch.arange(1, num_steps + 1, device=X.device
                             ).repeat(batch_size, 1)
        else:
            dec_valid_lens = None

        # Masked Self-Attention + Add&Norm
        X2 = self.attention1(X, key_values, key_values, dec_valid_lens)
        Y  = self.addnorm1(X, X2)
        # Cross-Attention + Add&Norm
        Y2 = self.attention2(Y, enc_outputs, enc_outputs, enc_valid_lens)
        Z  = self.addnorm2(Y, Y2)
        # FFN + Add&Norm
        return self.addnorm3(Z, self.ffn(Z)), state


In [ ]:
# ── Transformer Decoder ──
class TransformerDecoder(d2l.AttentionDecoder):
    def __init__(self, vocab_size, key_size, query_size, value_size,
                 num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, dropout):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential(*[
            DecoderBlock(key_size, query_size, value_size, num_hiddens,
                         norm_shape, ffn_num_input, ffn_num_hiddens,
                         num_heads, dropout, i)
            for i in range(num_layers)
        ])
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, enc_valid_lens, *args):
        return [enc_outputs, enc_valid_lens, [None] * self.num_layers]

    def forward(self, X, state):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        self._attention_weights = [[None] * len(self.blks) for _ in range(2)]
        for i, blk in enumerate(self.blks):
            X, state = blk(X, state)
            self._attention_weights[0][i] = blk.attention1.attention.attention_weights
            self._attention_weights[1][i] = blk.attention2.attention.attention_weights
        return self.dense(X), state

    @property
    def attention_weights(self):
        return self._attention_weights


In [ ]:
# ── 训练 Transformer（英译法） ──
num_hiddens, num_layers, dropout = 32, 2, 0.1
batch_size, num_steps = 64, 10
ffn_num_hiddens, num_heads = 64, 4
lr, num_epochs = 0.005, 200

train_iter, src_vocab, tgt_vocab = d2l.load_data_nmt(batch_size, num_steps)

encoder = TransformerEncoder(
    len(src_vocab), num_hiddens, num_hiddens, num_hiddens, num_hiddens,
    [num_hiddens], num_hiddens, ffn_num_hiddens, num_heads, num_layers, dropout)
decoder = TransformerDecoder(
    len(tgt_vocab), num_hiddens, num_hiddens, num_hiddens, num_hiddens,
    [num_hiddens], num_hiddens, ffn_num_hiddens, num_heads, num_layers, dropout)

net = d2l.EncoderDecoder(encoder, decoder)
d2l.train_seq2seq(net, train_iter, lr, num_epochs, tgt_vocab, device)


In [ ]:
# ── 预测与注意力可视化 ──
engs = ['go .', 'i lost .', 'he's calm .', 'i'm home .']
fras = ['va !',  'j'ai perdu .', 'il est calme .', 'je suis chez moi .']

for eng, fra in zip(engs, fras):
    translation, dec_attention_weight_seq = d2l.predict_seq2seq(
        net, eng, src_vocab, tgt_vocab, num_steps, device, save_attention_weights=True)
    print(f'{eng} => {translation}, BLEU={d2l.bleu(translation, fra, k=2):.3f}')

# 可视化编码器自注意力（第1个块）
enc_attention_weights = torch.cat(
    encoder.attention_weights, dim=0)
d2l.show_heatmaps(
    enc_attention_weights.cpu().reshape(-1, 1, num_steps, num_steps),
    xlabel='Key 位置', ylabel='Query 位置',
    titles=[f'Head {i+1}' for i in range(num_heads)],
    figsize=(7, 3.5))


## 8. 总结与对比

### 注意力评分函数对比

| 评分函数 | 适用场景 | 公式 | 参数量 |
|---|---|---|---|
| 加性注意力 | $q\_dim \neq k\_dim$ | $\mathbf{w}_v^\top \tanh(\mathbf{W}_q \mathbf{q} + \mathbf{W}_k \mathbf{k})$ | $O(qh + kh + h)$ |
| 缩放点积 | $q\_dim = k\_dim = d$ | $\mathbf{q}^\top \mathbf{k} / \sqrt{d}$ | $O(1)$（无参） |

### 各模块作用总结

| 模块 | 作用 |
|---|---|
| Masked Softmax | 忽略填充（padding）位置的注意力 |
| Bahdanau 注意力 | seq2seq 中动态对齐源序列 |
| 多头注意力 | 并行多子空间，捕获多样依赖 |
| 位置编码 | 为置换不变的自注意力注入位置信息 |
| Add & Norm | 残差连接 + LayerNorm，稳定深层训练 |
| Position-wise FFN | 对每个位置独立变换，增加非线性 |
| 掩蔽自注意力 | 解码器防止看到未来，保证自回归 |
| 交叉注意力 | 解码器查询编码器，实现 seq2seq 对齐 |

### Transformer 优势

1. **并行化**：所有位置同时计算，训练速度远超 RNN
2. **长程依赖**：$O(1)$ 最大路径长度，直接建模任意距离依赖
3. **可解释性**：注意力权重可以可视化，理解模型关注点
4. **可扩展性**：易于通过增加层数/头数/维度扩大模型容量

### 局限性

- 计算复杂度 $O(n^2d)$，超长序列（如文档级）计算开销大
- 需要大量训练数据才能发挥优势
- 位置编码的设计方案（绝对 vs 相对 vs 可学习）仍是活跃研究方向


In [ ]:
# ── 各模块参数量统计 ──
configs = {
    'AdditiveAttention': AdditiveAttention(key_size=64, query_size=64,
                                           num_hiddens=64, dropout=0),
    'DotProductAttention': DotProductAttention(dropout=0),
    'MultiHeadAttention (8头)': MultiHeadAttention(
        64, 64, 64, 64, num_heads=8, dropout=0),
    'PositionWiseFFN': PositionWiseFFN(64, 256, 64),
    'EncoderBlock': EncoderBlock(64, 64, 64, 64, [64], 64, 256, 8, dropout=0),
}

print(f"{'模块':<30} {'参数量':>10}")
print("-" * 42)
for name, module in configs.items():
    params = sum(p.numel() for p in module.parameters())
    print(f"{name:<30} {params:>10,}")
